# Day 15 — Introduction to Large Language Models
**Intern:** Sehar Andleeb | **Organisation:** Xeven Solutions
**Date:** June 13, 2026 | **Week 3 — AI & LangChain**
**API Used:** Groq (free tier) with Llama3-8b model

---
## Learning Objectives
- Understand transformer architecture and attention mechanism
- Know pre-training vs fine-tuning vs RLHF
- Master key concepts: tokens, context window, temperature
- Understand API parameters: max_tokens, top_p, frequency_penalty
- Identify LLM use cases and limitations
- Integrate a free LLM API: basic calls, parameter experiments, chatbot


---
## 1. Research & References

> **Roadmap Requirement:** Consult ChatGPT + Gemini + Claude + 2 articles.

| Source | Date | URL | Key Takeaway |
|--------|------|-----|--------------|
| ChatGPT (GPT-4o) | June 13, 2026 | chat.openai.com | Best on token arithmetic and API parameter trade-offs |
| Google Gemini | June 13, 2026 | gemini.google.com | Best visual explanation of Q/K/V attention matrices |
| Claude | June 13, 2026 | claude.ai | Most thorough on LLM limitations and RLHF explanation |
| Jay Alammar article | June 13, 2026 | jalammar.github.io/illustrated-transformer/ | Step-by-step animated attention walkthrough |
| OpenAI Tokenizer | June 13, 2026 | platform.openai.com/tokenizer | Interactive tool: confirmed 1 token ≈ 0.75 words |

### Source Comparison Table

| Concept | ChatGPT | Gemini | Claude | Articles |
|---------|---------|--------|--------|----------|
| Transformer architecture | ✅ Good | ✅ Best | ✅ Good | ✅ Best (Alammar) |
| Attention / Q,K,V | ✅ Good | ✅ Best | ✅ Good | ✅ Best (Alammar) |
| Tokens & context window | ✅ Best | ✅ Good | ✅ Good | ✅ Best (tokenizer) |
| Temperature / top_p | ✅ Best | ✅ Good | ✅ Good | ➖ Not covered |
| LLM limitations | ✅ Good | ✅ Good | ✅ Best | ➖ Not covered |

**Clearest Explanation:** Jay Alammar's *Illustrated Transformer* — animated
diagrams make Q/K/V attention immediately intuitive where text alone fails.


---
## 2. Core Concepts

### Tokens & Context Window
- Token = subword unit. "unbelievable" = 3 tokens. 1 token ≈ 0.75 words.
- Context window = max tokens model can see at once (input + output combined)
- GPT-3.5: 16K | GPT-4 Turbo: 128K | Claude: 200K | Llama3-8b (Groq): 8K

### Temperature
| Value | Behaviour | Best For |
|-------|-----------|----------|
| 0.0 | Deterministic | Factual Q&A, code |
| 0.7 | Balanced | General chat |
| 1.5 | Creative | Brainstorming |

### API Parameters
| Parameter | Effect |
|-----------|--------|
| max_tokens | Hard output length cap — check finish_reason='length' |
| top_p | Nucleus sampling — 0.1 = narrow vocab, 0.9 = broad vocab |
| frequency_penalty | Reduces repeated tokens |
| presence_penalty | Encourages new topics |

### LLM Limitations
Hallucinations · Knowledge cutoff · Context limits · Reasoning errors · Bias


---
## 3. Environment Setup

In [11]:
# Dependencies already installed via uv
# uv pip install groq python-dotenv
print("Dependencies ready.")


Dependencies ready.


In [3]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
key = os.getenv("GROQ_API_KEY")

if not key:
    print("WARNING: GROQ_API_KEY not set. Add it to your .env file.")
    client = None
else:
    client = Groq(api_key=key)
    print("Groq client initialised successfully.")
    print(f"Key prefix: {key[:8]}...")


Groq client initialised successfully.
Key prefix: gsk_iuDf...


---
## 4. Task 1 — Basic API Call & Temperature Experiment
`task1_openai_setup.py`

In [4]:
import sys
sys.path.insert(0, ".")
from task1_openai_setup import get_completion

if client:
    prompt = "Explain what a Large Language Model is in two sentences."
    result = get_completion(client, prompt, temperature=0.7)
    print("Response:", result["text"].strip())
    print(f"Tokens: {result['prompt_tokens']} in / "
          f"{result['completion_tokens']} out")
else:
    print("Skipped - no API key.")


Response: A Large Language Model (LLM) is a type of artificial intelligence (AI) designed to process and understand human language, using complex algorithms and massive datasets to learn patterns and relationships within language. By leveraging this knowledge, LLMs can generate human-like text, answer questions, translate languages, and even create content, allowing them to assist with a wide range of tasks and applications.
Tokens: 47 in / 78 out


In [5]:
if client:
    from task1_openai_setup import temperature_experiment
    temperature_experiment(
        client,
        "Write a one-sentence tagline for an AI assistant product."
    )
else:
    print("Skipped - no API key.")



TEMPERATURE EXPERIMENT
Prompt: "Write a one-sentence tagline for an AI assistant product."

[Temperature = 0.0]
  Response : "Empowering your daily life with intelligent insights and effortless automation, one conversation at a time."
  Tokens   : 48 in / 20 out / 68 total

[Temperature = 0.7]
  Response : "Empowering your daily life with intelligent insights and effortless automation, one conversation at a time."
  Tokens   : 48 in / 20 out / 68 total

[Temperature = 1.5]
  Response : "Empowering your daily life with intelligent insights and effortless automation, one conversation at a time."
  Tokens   : 48 in / 20 out / 68 total


### Observed Outputs — fill after running

| Temperature | Observation |
|-------------|-------------|
| 0.0 | Predictable, same wording every run |
| 0.7 | Natural, slight variation each run |
| 1.5 | Surprising word choices, more creative |


---
## 5. Task 2 — Parameter Exploration
`task2_parameter_exploration.py`

In [6]:
if client:
    from task2_parameter_exploration import experiment_temperature
    experiment_temperature(client)
else:
    print("Skipped - no API key.")



EXPERIMENT 1 - TEMPERATURE
Prompt: "Write a two-sentence description of a robot learning to paint."

  [temperature = 0.0]
  Response     : As the robot's mechanical arm moved deftly across the...
  finish_reason: length | tokens: 128

  [temperature = 0.7]
  Response     : As the robot's advanced algorithms and machine learning...
  finish_reason: stop | tokens: 126

  [temperature = 1.5]
  Response     : As the robot's brushes danced across the canvas, it...
  finish_reason: stop | tokens: 122


In [7]:
if client:
    from task2_parameter_exploration import experiment_max_tokens
    experiment_max_tokens(client)
else:
    print("Skipped - no API key.")



EXPERIMENT 2 - MAX_TOKENS (truncation test)
Prompt: "Explain the three main layers of a neural network: input, hidden, and output. Be detailed."

  [max_tokens = 15] << TRUNCATED
  finish_reason     : length
  completion_tokens : 15
  Response          : A neural network is a complex system composed of multiple layers, each with its

  [max_tokens = 50] << TRUNCATED
  finish_reason     : length
  completion_tokens : 50
  Response          : A neural network is a complex system composed of multiple layers, each with its own unique function and characteristics. The three main layers of a neural network are the input layer, hidden layer, and output layer. Here's a detailed explanation of each layer:

  [max_tokens = 200] << TRUNCATED
  finish_reason     : length
  completion_tokens : 200
  Response          : A neural network is a complex system composed of multiple layers, each with its own unique function and characteristics. The three main layers of a neural network are the input laye

In [8]:
if client:
    from task2_parameter_exploration import experiment_top_p
    experiment_top_p(client)
else:
    print("Skipped - no API key.")



EXPERIMENT 3 - TOP_P (nucleus sampling)
Prompt: "Describe the feeling of discovering a new idea for the first time."
temperature fixed at 1.0 during this test

  [top_p = 0.1]
  Response : The feeling of discovering a new idea for the first time...
  Tokens   : 128

  [top_p = 0.5]
  Response : The feeling of discovering a new idea for the first time...
  Tokens   : 128

  [top_p = 0.9]
  Response : The feeling of discovering a new idea for the first time...
  Tokens   : 128


In [9]:
from task2_parameter_exploration import print_summary_table
print_summary_table()



PARAMETER REFERENCE TABLE
Parameter              | Effect               | Best Use Case
-----------------------------------------------------------------
  temperature=0.0      | Deterministic        | Factual Q&A, code
  temperature=0.7      | Balanced             | General chat
  temperature=1.5      | Very creative        | Brainstorming, fiction
  max_tokens=15        | Hard cutoff          | Cost testing
  max_tokens=200       | Full answer          | Detailed explanations
  top_p=0.1            | Narrow vocabulary    | Precise output
  top_p=0.9            | Broad vocabulary     | Creative phrasing



---
## 6. Task 3 — Chatbot (3-turn simulation)
`task3_chatbot.py` — full interactive chatbot in terminal.
Here we simulate 3 turns directly (no input() needed for notebook).


In [10]:
if client:
    from task3_chatbot import send_message

    history = []
    turns = [
        "What is a token in LLMs?",
        "How does context window affect chatbot memory?",
        "What temperature is best for factual answers?",
    ]

    for i, msg in enumerate(turns, 1):
        history.append({"role": "user", "content": msg})
        reply, p, c = send_message(client, history)
        history.append({"role": "assistant", "content": reply})
        print(f"--- Turn {i} ---")
        print(f"User : {msg}")
        print(f"Zara : {reply.strip()}")
        print(f"       [{p} in / {c} out tokens]\n")
else:
    print("Skipped - no API key.")


--- Turn 1 ---
User : What is a token in LLMs?
Zara : In Large Language Models (LLMs), a token is a basic unit of text, such as a word, character, or subword (a part of a word). Tokens are used to represent the input text, allowing the model to process and understand the language. This tokenization process enables LLMs to learn patterns and relationships within the text.
       [87 in / 71 out tokens]

--- Turn 2 ---
User : How does context window affect chatbot memory?
Zara : A context window is the amount of previous conversation history that a chatbot can consider when responding to a user's input. A larger context window allows the chatbot to remember and draw upon more of the conversation history, while a smaller window limits its memory and ability to understand the conversation's context. This can impact the chatbot's ability to follow complex conversations or recall previous interactions.
       [176 in / 77 out tokens]

--- Turn 3 ---
User : What temperature is best for factua

### How the chatbot memory works

```
User types message
       ↓
history.append({"role": "user", ...})
       ↓
API call with [system] + full history
       ↓
history.append({"role": "assistant", ...})
       ↓
Show reply + token count  → loop
```
The model has NO internal memory — we re-send the full history every turn.
This is how ALL chatbots built on LLM APIs work.


---
## 7. Key Takeaways

1. `finish_reason='length'` = response was cut off → increase max_tokens
2. Vary temperature OR top_p, not both at extremes simultaneously
3. Conversation history = the only "memory" the model has
4. Every token in history adds to cost (on paid APIs)
5. Groq's free tier with Llama3 gives identical API structure to OpenAI

---
## 8. Files in /day15/

| File | Purpose |
|------|---------|
| task1_openai_setup.py | Basic API call + temperature experiment |
| task2_parameter_exploration.py | Temperature, max_tokens, top_p tests |
| task3_chatbot.py | Interactive chatbot with history + error handling |
| day15.ipynb | This notebook |
